In [1]:
import xarray as xr
from obstore.store import from_url

from virtualizarr import open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry
from distributed import Client
# from srm.utils import lon_to_180

In [3]:
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-gzdql.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-gzdql.dask.host/jupyter/proxy/8787/status,Workers: 4
Total threads: 8,Total memory: 30.11 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38059,Workers: 0
Dashboard: https://cluster-gzdql.dask.host/jupyter/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33739,Total threads: 2
Dashboard: https://cluster-gzdql.dask.host/jupyter/proxy/35075/status,Memory: 7.53 GiB
Nanny: tcp://127.0.0.1:39537,


In [2]:
ds = xr.open_dataset(
    "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.TS.20350101-20441231.nc",
    chunks={},
    engine="h5netcdf",
)
ds

<xarray.Dataset> Size: 808MB
Dimensions:       (lat: 192, lev: 70, ilev: 71, time: 3650, nbnd: 2, lon: 288)
Coordinates:
  * lat           (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon           (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
  * lev           (lev) float64 560B 5.96e-06 9.827e-06 1.62e-05 ... 976.3 992.6
  * ilev          (ilev) float64 568B 4.5e-06 7.42e-06 1.223e-05 ... 985.1 1e+03
  * time          (time) object 29kB 2035-01-01 00:00:00 ... 2044-12-31 00:00:00
Dimensions without coordinates: nbnd
Data variables: (12/26)
    gw            (lat) float64 2kB dask.array<chunksize=(192,), meta=np.ndarray>
    hyam          (lev) float64 560B dask.array<chunksize=(70,), meta=np.ndarray>
    hybm          (lev) float64 560B dask.array<chunksize=(70,), meta=np.ndarray>
    P0            float64 8B ...
    hyai          (ilev) float64 568B dask.array<chunksize=(71,), meta=np.ndarray>
    hybi          (ilev) float64 568B dask.array<chunksize=(71,), meta=np.ndarray>
    ...            ...
    n2ovmr        (time) float64 29kB dask.array<chunksize=(512,), meta=np.ndarray>
    f11vmr        (time) float64 29kB dask.array<chunksize=(512,), meta=np.ndarray>
    f12vmr        (time) float64 29kB dask.array<chunksize=(512,), meta=np.ndarray>
    sol_tsi       (time) float64 29kB dask.array<chunksize=(512,), meta=np.ndarray>
    nsteph        (time) int32 15kB dask.array<chunksize=(1024,), meta=np.ndarray>
    TS            (time, lat, lon) float32 807MB dask.array<chunksize=(1, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006
    logname:           geostrat
    host:              cheyenne1
    initial_file:      b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.c...
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [3]:
bucket = "s3://ncar-cesm2-arise/"
path1 = "ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.TS.20350101-20441231.nc"
path2 = "ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECC.20350101-20441231.nc"
path3 = "ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.U.20350101-20441231.nc"

url1 = f"{bucket}/{path1}"
url2 = f"{bucket}/{path2}"
url3 = f"{bucket}/{path3}"

store = from_url(bucket, region="us-east-2", skip_signature=True)
registry = ObjectStoreRegistry({bucket: store})

In [4]:
drop_vars = [
    "gw",
    "hyam",
    "hybm",
    "P0",
    "hyai",
    "hybi",
    "ndbase",
    "nsbase",
    "nbdate",
    "nbsec",
    "mdt",
    "date",
    "datesec",
    "time_bnds",
    "date_written",
    "time_written",
    "ndcur",
    "nscur",
    "co2vmr",
    "ch4vmr",
    "n2ovmr",
    "f11vmr",
    "f12vmr",
    "sol_tsi",
    "nsteph",
]
parser = HDFParser(drop_variables=drop_vars)

# vds = open_virtual_dataset(
#   url=f"{bucket}/{path1}",
#   parser=parser,
#   registry=registry,
#   # drop_variables=drop_vars,
#   loadable_variables =['lat', 'lev', 'ilev', 'time', 'nbnd', 'lon'],
#     decode_times=True,

# )

In [5]:
combined_vds = open_virtual_mfdataset(
    [url1, url2, url3],
    registry=registry,
    parser=parser,
    combine="by_coords",
    compat="override",
    combine_attrs="drop_conflicts",
    loadable_variables=["lat", "lev", "ilev", "time", "nbnd", "lon"],
    parallel="lithops",
)

ModuleNotFoundError: No module named 'lithops'

In [8]:
combined_vds

<xarray.Dataset> Size: 58GB
Dimensions:  (lat: 192, lon: 288, lev: 70, ilev: 71, time: 3650)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * lev      (lev) float64 560B 5.96e-06 9.827e-06 1.62e-05 ... 976.3 992.6
  * ilev     (ilev) float64 568B 4.5e-06 7.42e-06 1.223e-05 ... 985.1 1e+03
  * time     (time) object 29kB 2035-01-01 00:00:00 ... 2044-12-31 00:00:00
Data variables:
    TS       (time, lat, lon) float32 807MB ManifestArray<shape=(3650, 192, 2...
    PRECC    (time, lat, lon) float32 807MB ManifestArray<shape=(3650, 192, 2...
    U        (time, lev, lat, lon) float32 57GB ManifestArray<shape=(3650, 70...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006
    logname:           geostrat
    host:              cheyenne1
    initial_file:      b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.c...
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1